# AFC-FullBench: full-episode training, testing, and visualization

This notebook runs the full-episode alarm flood classification benchmark on the TEP and FCC datasets. It trains every configured classifier on complete training episodes, evaluates on complete held-out episodes, and creates summary figures. No online prefixes, perturbations, or robustness calculations are used here.

Before running the notebook, place the datasets under `data/tep/` and `data/fcc/` following the class-folder layout described in the README.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

# Resolve the repository root when the notebook is executed from notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from afc_fullbench.data import AlarmDataset, load_alarm_series_dataset
from afc_fullbench.evaluation import run_cross_validation
from afc_fullbench.plotting import plot_confusion_matrices, plot_summary_bar

FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Set RUN_CV=False if CSV outputs already exist and only figures/tables should be regenerated.
RUN_CV = True

DATASET_CONFIGS = {
    "TEP": PROJECT_ROOT / "configs" / "tep.yaml",
    "FCC": PROJECT_ROOT / "configs" / "fcc.yaml",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures will be written to: {FIGURES_DIR}")

## Helper functions

The helper functions below keep the actual TEP and FCC execution cells short. They deliberately use the same package functions as the command-line interface, so notebook and CLI results remain aligned.

In [ ]:
def load_yaml(path: str | Path) -> dict[str, Any]:
    """Load one YAML configuration file and return a plain dictionary."""
    with open(path, "r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    if config is None:
        raise ValueError(f"Configuration file is empty: {path}")
    return dict(config)


def load_dataset_from_config(config: dict[str, Any]) -> AlarmDataset:
    """Load one class-folder alarm-series dataset from a benchmark config."""
    data_cfg = config.get("data", {})
    root = PROJECT_ROOT / data_cfg.get("root", data_cfg.get("path", "data/tep"))
    return load_alarm_series_dataset(
        root,
        max_time_steps=data_cfg.get("max_time_steps", data_cfg.get("max_length")),
        dtype=data_cfg.get("dtype", "float32"),
    )


def run_dataset_pipeline(dataset_label: str, config_path: str | Path) -> dict[str, pd.DataFrame]:
    """Run or load the full-episode benchmark for one dataset.

    Parameters
    ----------
    dataset_label:
        Human-readable dataset identifier, for example ``"TEP"`` or ``"FCC"``.
    config_path:
        YAML configuration file containing data, CV, parallelization, and model settings.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Dictionary containing ``fold_metrics``, ``predictions``,
        ``confusion_matrices``, ``summary``, and auxiliary metadata tables.
    """
    config = load_yaml(config_path)
    output_dir = PROJECT_ROOT / config.get("output_dir", f"results/{dataset_label.lower()}_full_episode")
    output_dir.mkdir(parents=True, exist_ok=True)

    if RUN_CV:
        dataset = load_dataset_from_config(config)
        cv_cfg = config.get("cv", {})
        parallel_cfg = config.get("parallel", {})

        print(
            f"Running {dataset_label}: "
            f"{dataset.X.shape[0]} episodes, {dataset.X.shape[1]} tags, "
            f"{dataset.X.shape[2]} time steps, {len(dataset.class_names)} classes"
        )

        outputs = run_cross_validation(
            dataset,
            model_configs=config.get("models", []),
            n_splits=int(cv_cfg.get("n_splits", 5)),
            n_repeats=int(cv_cfg.get("n_repeats", 1)),
            shuffle=bool(cv_cfg.get("shuffle", True)),
            random_state=int(cv_cfg.get("random_state", config.get("random_seed", 42))),
            n_jobs=int(parallel_cfg.get("n_jobs", 1)),
            backend=str(parallel_cfg.get("backend", "loky")),
            inner_max_num_threads=parallel_cfg.get("inner_max_num_threads", 1),
            pre_dispatch=parallel_cfg.get("pre_dispatch", "2*n_jobs"),
            output_dir=output_dir,
        )
    else:
        print(f"Loading existing outputs for {dataset_label} from {output_dir}")
        outputs = {
            "fold_metrics": pd.read_csv(output_dir / "fold_metrics.csv"),
            "predictions": pd.read_csv(output_dir / "predictions.csv"),
            "confusion_matrices": pd.read_csv(output_dir / "confusion_matrices.csv"),
            "summary": pd.read_csv(output_dir / "summary.csv"),
        }

    # Auxiliary tables are needed for plotting and interpretation.
    outputs["classes"] = pd.read_csv(output_dir / "classes.csv")
    outputs["metadata"] = pd.read_csv(output_dir / "metadata.csv")
    outputs["output_dir"] = pd.DataFrame({"path": [str(output_dir)]})
    return outputs


def make_dataset_figures(dataset_label: str, outputs: dict[str, pd.DataFrame]) -> list[Path]:
    """Create standard summary and pooled confusion-matrix figures for one dataset."""
    output_paths: list[Path] = []
    output_paths.append(
        plot_summary_bar(
            outputs["summary"],
            FIGURES_DIR,
            dataset_label=dataset_label,
            metric="accuracy",
        )
    )
    output_paths.extend(
        plot_confusion_matrices(
            outputs["predictions"],
            outputs["classes"],
            FIGURES_DIR,
            dataset_label=dataset_label,
            normalize="true",
        )
    )
    return output_paths


def display_core_summary(dataset_label: str, outputs: dict[str, pd.DataFrame]) -> None:
    """Display the main cross-validation summary columns for one dataset."""
    cols = [
        "method",
        "n_units",
        "accuracy_mean",
        "accuracy_std",
        "balanced_accuracy_mean",
        "balanced_accuracy_std",
        "macro_f1_mean",
        "macro_f1_std",
        "fit_seconds_mean",
        "predict_seconds_mean",
    ]
    existing = [col for col in cols if col in outputs["summary"].columns]
    print(f"\n{dataset_label} summary")
    display(outputs["summary"][existing])

## Run TEP

In [ ]:
tep_result = run_dataset_pipeline("TEP", DATASET_CONFIGS["TEP"])
display_core_summary("TEP", tep_result)
tep_figures = make_dataset_figures("TEP", tep_result)
tep_figures

## Run FCC

In [ ]:
fcc_result = run_dataset_pipeline("FCC", DATASET_CONFIGS["FCC"])
display_core_summary("FCC", fcc_result)
fcc_figures = make_dataset_figures("FCC", fcc_result)
fcc_figures

## Combined summary table

The combined table is useful for quick comparisons across datasets. It is saved as a CSV file under `results/full_episode_combined_summary.csv`.

In [ ]:
def summary_with_dataset(dataset_label: str, outputs: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Attach a dataset label to one summary table."""
    table = outputs["summary"].copy()
    table.insert(0, "dataset", dataset_label)
    return table

combined_summary = pd.concat(
    [summary_with_dataset("TEP", tep_result), summary_with_dataset("FCC", fcc_result)],
    ignore_index=True,
)
combined_summary_path = RESULTS_DIR / "full_episode_combined_summary.csv"
combined_summary.to_csv(combined_summary_path, index=False)
print(f"Saved combined summary to: {combined_summary_path}")

display(combined_summary)

## Combined accuracy visualization

In [ ]:
def plot_combined_accuracy(summary: pd.DataFrame, output_path: Path) -> Path:
    """Plot method-wise accuracy means for both datasets in one compact figure."""
    pivot = summary.pivot(index="method", columns="dataset", values="accuracy_mean")
    ax = pivot.plot(kind="bar", figsize=(8.0, 3.8), ylim=(0.0, 1.0))
    ax.set_ylabel("Accuracy")
    ax.set_xlabel("AFC method")
    ax.grid(axis="y", linewidth=0.4, alpha=0.4)
    ax.legend(title="Dataset")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, bbox_inches="tight")
    plt.savefig(output_path.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    return output_path

combined_accuracy_figure = plot_combined_accuracy(
    combined_summary,
    FIGURES_DIR / "combined_accuracy_summary.pdf",
)
combined_accuracy_figure

## Generated figure files

In [ ]:
all_figures = sorted(FIGURES_DIR.glob("*.pdf"))
for path in all_figures:
    print(path.relative_to(PROJECT_ROOT))